In [2]:
from decimal import Decimal
from datetime import datetime
from typing import TypedDict
from enum import Enum


class OperationType(str, Enum):
    DEPOSIT = "deposit"
    WITHDRAW = "withdraw"
    INTEREST = "interest"


class OperationStatus(str, Enum):
    SUCCESS = "success"
    FAIL = "fail"


class Operation(TypedDict):
    type: OperationType
    status: OperationStatus
    amount: Decimal
    current_balance: Decimal
    timestamp: datetime
    comment: str


In [3]:
class AccountError(Exception):
    """Base exception for account-related errors."""


class AccountValidationError(AccountError):
    """Raised when input data is invalid."""


In [4]:
from decimal import Decimal
from dataclasses import dataclass


@dataclass(frozen=True)
class Transaction:
    account_number: str
    account_type: str
    date: datetime
    operation: OperationType
    amount: Decimal
    balance_after: Decimal
    status: OperationStatus

In [5]:
class AccountType(str, Enum):
    CHECKING = "checking"
    SAVINGS = "savings"

In [6]:
from dataclasses import dataclass


@dataclass(frozen=True)
class TransactionImportError:
    row_number: int
    message: str

In [7]:
from dataclasses import dataclass


@dataclass(frozen=True)
class TransactionImportResult:
    loaded: int
    skipped: int
    errors: tuple[TransactionImportError, ...]

In [8]:
from pathlib import Path
from decimal import Decimal
from zoneinfo import ZoneInfo
import re


class Account:
    account_type: AccountType = None
    _account_counter: int = 1000
    _MOSCOW_TZ = ZoneInfo("Europe/Moscow")

    _HOLDER_PATTERN = re.compile(
        r"^(?:[А-ЯЁ][а-яё]+ [А-ЯЁ][а-яё]+|[A-Z][a-z]+ [A-Z][a-z]+)$"
    )

    def __init__(self, account_holder: str, balance: Decimal | float = Decimal("0")) -> None:
        balance = Decimal(str(balance))

        if balance < 0:
            raise AccountValidationError("balance cannot be negative")

        self.validate_holder_name(account_holder)

        self._balance: Decimal = balance
        self.holder: str = account_holder
        self.account_number: str = self.generate_account_number()
        self.operations_history: list[Operation] = []

    @classmethod
    def generate_account_number(cls) -> str:
        account_number = str(cls._account_counter)
        Account._account_counter += 1
        return "ACC-" + account_number

    @classmethod
    def validate_holder_name(cls, name: str) -> None:
        if not isinstance(name, str):
            raise AccountValidationError("holder name must be a string")

        if not cls._HOLDER_PATTERN.fullmatch(name):
            raise AccountValidationError("holder name must follow 'John Parker' OR 'Иван Иванов' format")

    def deposit(self, amount: Decimal | float) -> None:
        amount = Decimal(str(amount))

        if amount <= 0:
            raise AccountValidationError("amount must be positive")

        self._balance += amount

        operation: Operation = Operation(
            type=OperationType.DEPOSIT,
            status=OperationStatus.SUCCESS,
            amount=amount,
            current_balance=self._balance,
            timestamp=datetime.now(self._MOSCOW_TZ),
            comment=""
        )

        self.operations_history.append(operation)

    def withdraw(self, amount: Decimal | float) -> None:
        amount = Decimal(str(amount))

        comment = ""
        status = OperationStatus.SUCCESS

        if amount <= 0:
            raise AccountValidationError("amount must be positive")

        if amount > self._balance:
            comment = "amount exceeds balance"
            status = OperationStatus.FAIL

        if status == OperationStatus.SUCCESS:
            self._balance -= amount

        operation: Operation = Operation(
            type=OperationType.WITHDRAW,
            status=status,
            amount=amount,
            current_balance=self._balance,
            timestamp=datetime.now(self._MOSCOW_TZ),
            comment=comment
        )

        self.operations_history.append(operation)

    def get_balance(self) -> Decimal:
        return self._balance

    def get_history(self) -> list[Operation]:
        return self.operations_history.copy()

    def get_account_counter(self) -> int:
        return self._account_counter

    def get_largest_operations(self, number: int) -> list[Operation]:
        if number <= 0:
            raise AccountValidationError(
                "n must be positive"
            )

        sorted_operations = sorted(
            self.operations_history,
            key=lambda operation: operation["amount"],
            reverse=True
        )

        return sorted_operations[:number]


    def apply_transaction(self, transaction: Transaction) -> None:
        if transaction.account_number != self.account_number:
            raise TransactionValidationError(
                "Transaction belongs to another account"
            )

        if transaction.account_type != self.account_type:
            raise TransactionValidationError(
                "Transaction account type does not match"
            )

        if transaction.amount <= 0:
            raise TransactionValidationError(
                "Transaction amount must be positive"
            )

        expected_status, expected_balance = (
            self._calculate_transaction_result(transaction)
        )

        if transaction.status != expected_status:
            raise TransactionValidationError(
                "Transaction status does not match the expected result"
            )

        self._balance = expected_balance

        operation = Operation(
            type=transaction.operation,
            status=transaction.status,
            amount=transaction.amount,
            current_balance=expected_balance,
            timestamp=transaction.date,
            comment="",
        )

        self.operations_history.append(operation)

    def _calculate_transaction_result(
            self,
            transaction: Transaction
    ) -> tuple[OperationStatus, Decimal]:
        if transaction.operation == OperationType.DEPOSIT:
            expected_balance = self._balance + transaction.amount
            return OperationStatus.SUCCESS, expected_balance

        if transaction.operation == OperationType.WITHDRAW:
            if transaction.amount > self._balance:
                return OperationStatus.FAIL, self._balance

            expected_balance = self._balance - transaction.amount
            return OperationStatus.SUCCESS, expected_balance

        raise TransactionValidationError(
            f"Operation {transaction.operation.value} "
            "is not supported by this account"
        )

    def load_transactions(
    self,
    path: str | Path,
    ) -> TransactionImportResult:
        """Load and apply account transactions from a file."""
        importer = TransactionImporter(
            reader=TransactionFileReader(),
            validator=TransactionValidator(),
        )

        return importer.import_transactions(
            path=path,
            account=self,
        )

    def clean_history(
        self,
        path: str | Path,
    ) -> TransactionImportResult:
        """Validate and load valid transactions into account history."""
        return self.load_transactions(path)

In [9]:
class CheckingAccount(Account):
    account_type: str = AccountType.CHECKING


In [10]:
from decimal import Decimal


class SavingsAccount(Account):
    account_type: str = AccountType.SAVINGS

    _APPLY_INTEREST_COMMENT: str = (
        "Interest applied for savings account"
    )

    _MAX_RATE: Decimal = Decimal("1000.0")

    def apply_interest(self, rate: Decimal | float) -> None:
        rate = Decimal(str(rate))

        if rate <= 0 or rate > self._MAX_RATE:
            raise AccountValidationError("Incorrect rate value. Rate must be positive and less than 1000%")

        interest = self._balance * rate / Decimal("100")

        self._balance += interest

        operation: Operation = Operation(
            type=OperationType.INTEREST,
            status=OperationStatus.SUCCESS,
            amount=interest,
            current_balance=self._balance,
            timestamp=datetime.now(self._MOSCOW_TZ),
            comment=self._APPLY_INTEREST_COMMENT
        )

        self.operations_history.append(operation)

    def withdraw(self, amount: Decimal | float) -> None:
        amount = Decimal(str(amount))

        status = OperationStatus.SUCCESS
        comment = ""

        if amount <= 0:
            raise AccountValidationError("amount must be positive")

        if amount > self._balance:
            comment = "amount exceeds balance"
            status = OperationStatus.FAIL
        elif amount > self._balance / Decimal("2"):
            comment = "amount exceeds 50% of current balance"
            status = OperationStatus.FAIL

        if status == OperationStatus.SUCCESS:
            self._balance -= amount

        operation: Operation = Operation(
            type=OperationType.WITHDRAW,
            status=status,
            amount=amount,
            current_balance=self._balance,
            timestamp=datetime.now(self._MOSCOW_TZ),
            comment=comment
        )

        self.operations_history.append(operation)

    def _calculate_transaction_result(
            self,
            transaction: Transaction,
    ) -> tuple[OperationStatus, Decimal]:
        if transaction.operation == OperationType.INTEREST:
            expected_balance = self._balance + transaction.amount
            return OperationStatus.SUCCESS, expected_balance

        if transaction.operation == OperationType.WITHDRAW:
            if transaction.amount > self._balance:
                return OperationStatus.FAIL, self._balance

            if transaction.amount > self._balance / Decimal("2"):
                return OperationStatus.FAIL, self._balance

            expected_balance = self._balance - transaction.amount
            return OperationStatus.SUCCESS, expected_balance

        return super()._calculate_transaction_result(transaction)

In [11]:
class TransactionLoaderError(Exception):
    """Base exception for transaction-loader-related errors."""


class JSONTransactionLoaderError(TransactionLoaderError):
    """Raised when JSON format data is invalid."""

class CSVTransactionLoaderError(TransactionLoaderError):
    """Raised when CSV format data is invalid."""

class TransactionValidationError(TransactionLoaderError):
    """Raised when a transaction contains invalid data."""

In [12]:
from typing import Any
from pathlib import Path
from abc import abstractmethod, ABC


class TransactionLoader(ABC):

    @abstractmethod
    def load(self, path: Path) -> list[dict[str, Any]]:
        """Load raw transactions from a file."""
        pass

In [13]:
import csv


class CsvTransactionLoader(TransactionLoader):

    def load(self, path: Path) -> list[dict[str, Any]]:
        with path.open("r", encoding="utf-8", newline="") as file:
            reader = csv.DictReader(file)
            return list(reader)

In [14]:
import json


class JsonTransactionLoader(TransactionLoader):

    def load(self, path: Path) -> list[dict[str, Any]]:
        with path.open("r", encoding="utf-8") as file:
            data = json.load(file)

        if not isinstance(data, list):
            raise JSONTransactionLoaderError("JSON transaction file must contain a list")

        return data

In [15]:
class TransactionLoaderFactory:

    _LOADERS: dict[str, type[TransactionLoader]] = {
        ".csv": CsvTransactionLoader,
        ".json": JsonTransactionLoader,
    }

    @classmethod
    def create(cls, path: Path) -> TransactionLoader:
        suffix = path.suffix.lower()

        try:
            loader_class = cls._LOADERS[suffix]
        except KeyError:
            raise TransactionLoaderError(f"Unsupported transaction file format: {suffix}")

        return loader_class()

In [16]:
class TransactionFileReader:

    def read(self, path: str | Path) -> list[dict[str, Any]]:
        path = Path(path)

        loader = TransactionLoaderFactory.create(path)

        return loader.load(path)

In [17]:
import re
from collections.abc import Mapping
from datetime import datetime
from decimal import Decimal, InvalidOperation
from typing import Any


class TransactionValidator:
    REQUIRED_FIELDS = {
        "account_number",
        "account_type",
        "date",
        "operation",
        "amount",
        "balance_after",
        "status",
    }

    ACCOUNT_NUMBER_PATTERN = re.compile(r"^ACC-\d+$")
    DATE_FORMAT = "%Y-%m-%d %H:%M:%S"

    def validate(self, raw_transaction: Mapping[str, Any]) -> Transaction:
        self._validate_required_fields(raw_transaction)

        return Transaction(
            account_number=self._parse_account_number(
                raw_transaction["account_number"]),
            account_type=self._parse_account_type(
                raw_transaction["account_type"]),
            date=self._parse_date(raw_transaction["date"]),
            operation=self._parse_operation(raw_transaction["operation"]),
            amount=self._parse_amount(raw_transaction["amount"]),
            balance_after=self._parse_amount(
                raw_transaction["balance_after"]),
            status=self._parse_status(raw_transaction["status"]),
        )

    def _validate_required_fields(
        self,
        transaction: Mapping[str, Any],
    ) -> None:
        missing_fields = self.REQUIRED_FIELDS - transaction.keys()

        if missing_fields:
            raise TransactionValidationError(
                f"Missing required fields: {sorted(missing_fields)}"
            )

    def _parse_account_number(self, value: Any) -> str:
        if not isinstance(value, str):
            raise TransactionValidationError("account_number must be a string")

        account_number = value.strip()

        if not self.ACCOUNT_NUMBER_PATTERN.fullmatch(account_number):
            raise TransactionValidationError(
                f"Invalid account number: {account_number}"
            )

        return account_number

    def _parse_account_type(self, value: Any) -> AccountType:
        if not isinstance(value, str):
            raise TransactionValidationError("account_type must be a string")

        normalized_value = value.strip().lower()

        try:
            return AccountType(normalized_value)
        except ValueError as error:
            raise TransactionValidationError(
                f"Invalid account type: {value}"
            ) from error

    def _parse_operation(self, value: Any) -> OperationType:
        if not isinstance(value, str):
            raise TransactionValidationError("operation must be a string")

        normalized_value = value.strip().lower()

        try:
            return OperationType(normalized_value)
        except ValueError as error:
            raise TransactionValidationError(
                f"Invalid operation: {value}"
            ) from error

    def _parse_status(self, value: Any) -> OperationStatus:
        if not isinstance(value, str):
            raise TransactionValidationError("status must be a string")

        normalized_value = value.strip().lower()

        try:
            return OperationStatus(normalized_value)
        except ValueError as error:
            raise TransactionValidationError(
                f"Invalid status: {value}"
            ) from error

    def _parse_amount(self, value: Any) -> Decimal:
        try:
            amount = Decimal(str(value))
        except (InvalidOperation, ValueError, TypeError) as error:
            raise TransactionValidationError(
                f"Invalid monetary value: {value}"
            ) from error

        if not amount.is_finite():
            raise TransactionValidationError(
                "Monetary value must be finite"
            )

        if amount < 0:
            raise TransactionValidationError(
                "Monetary value cannot be negative"
            )

        return amount

    def _parse_date(self, value: Any) -> datetime:
        if not isinstance(value, str):
            raise TransactionValidationError("date must be a string")

        normalized_value = value.strip()

        try:
            return datetime.strptime(normalized_value, self.DATE_FORMAT)
        except ValueError as error:
            raise TransactionValidationError(
                f"Invalid date: {value}"
            ) from error

In [18]:
from collections.abc import Mapping
from pathlib import Path
from typing import Any


class TransactionImporter:

    def __init__(
        self,
        reader: TransactionFileReader,
        validator: TransactionValidator,
    ) -> None:
        self._reader = reader
        self._validator = validator

    def import_transactions(
        self,
        path: str | Path,
        account: Account,
    ) -> TransactionImportResult:
        raw_transactions = self._reader.read(path)

        loaded = 0
        errors: list[TransactionImportError] = []

        for row_number, raw_transaction in enumerate(raw_transactions, start=2):
            if not self._belongs_to_account(raw_transaction, account):
                continue

            try:
                transaction = self._validator.validate(raw_transaction)
                account.apply_transaction(transaction)
            except TransactionValidationError as error:
                errors.append(
                    TransactionImportError(
                        row_number=row_number,
                        message=str(error),
                    )
                )
                continue

            loaded += 1

        return TransactionImportResult(
            loaded=loaded,
            skipped=len(errors),
            errors=tuple(errors),
        )

    def _belongs_to_account(
        self,
        raw_transaction: Mapping[str, Any],
        account: Account,
    ) -> bool:
        return raw_transaction.get("account_number") == account.account_number

    def _is_compatible_with_account(
        self,
        transaction: Transaction,
        account: Account,
    ) -> bool:
        return transaction.account_type.value == account.account_type

Дополнительно: написание unit-тестов для текущей реализации

In [19]:
import unittest

TRANSACTIONS_DIR = Path("Файлы_к_ДЗ_1")
CSV_FILE = TRANSACTIONS_DIR / "transactions_dirty.csv"
JSON_FILE = TRANSACTIONS_DIR / "transactions_dirty.json"

class TestAccount(unittest.TestCase):

    def test_create_account_with_zero_balance(self):
        account = Account("Ivan Ivanov")

        self.assertEqual(account.get_balance(), Decimal("0"))

    def test_create_account_with_initial_balance(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        self.assertEqual(account.get_balance(), Decimal("1000"))

    def test_create_account_with_negative_balance(self):
        with self.assertRaises(AccountValidationError):
            Account("Ivan Ivanov", Decimal("-100"))

    def test_account_holder(self):
        account = Account("Ivan Ivanov",Decimal("1000"))

        self.assertEqual(account.holder,"Ivan Ivanov")

    def test_account_number(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        self.assertRegex(account.account_number, r"^ACC-\d{4}$")

    def test_deposit(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        account.deposit(Decimal("500"))

        self.assertEqual(account.get_balance(), Decimal("1500"))

    def test_deposit_creates_operation(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        account.deposit(Decimal("500"))

        history = account.get_history()

        self.assertEqual(len(history), 1)

        operation = history[0]

        self.assertEqual(operation["type"], OperationType.DEPOSIT)

        self.assertEqual(operation["status"], OperationStatus.SUCCESS)

        self.assertEqual(operation["amount"], Decimal("500"))

        self.assertEqual(operation["current_balance"], Decimal("1500"))

    def test_deposit_negative_amount(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        with self.assertRaises(AccountValidationError):
            account.deposit(Decimal("-500"))

    def test_deposit_zero_amount(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        with self.assertRaises(AccountValidationError):
            account.deposit(Decimal("0"))

    def test_withdraw(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        account.withdraw(Decimal("300"))

        self.assertEqual(account.get_balance(), Decimal("700"))

    def test_withdraw_creates_operation(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        account.withdraw(Decimal("300"))

        operation = account.get_history()[0]

        self.assertEqual(operation["type"], OperationType.WITHDRAW)

        self.assertEqual(operation["status"], OperationStatus.SUCCESS)

        self.assertEqual(operation["amount"], Decimal("300"))

        self.assertEqual(operation["current_balance"], Decimal("700"))

    def test_withdraw_insufficient_balance(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        account.withdraw(Decimal("1500"))

        self.assertEqual(account.get_balance(), Decimal("1000"))

        operation = account.get_history()[0]

        self.assertEqual(operation["status"], OperationStatus.FAIL)

    def test_withdraw_negative_amount(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        with self.assertRaises(AccountValidationError):
            account.withdraw(Decimal("-100"))

    def test_withdraw_zero_amount(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        with self.assertRaises(AccountValidationError):
            account.withdraw(Decimal("0"))

    def test_history_contains_operations_in_order(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        account.deposit(Decimal("500"))
        account.withdraw(Decimal("200"))
        account.deposit(Decimal("300"))

        history = account.get_history()

        self.assertEqual(len(history), 3)

        self.assertEqual(history[0]["type"], OperationType.DEPOSIT)

        self.assertEqual(history[1]["type"], OperationType.WITHDRAW)

        self.assertEqual(history[2]["type"], OperationType.DEPOSIT)

    def test_get_history_returns_copy(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        account.deposit(Decimal("500"))

        history = account.get_history()
        history.clear()

        self.assertEqual(len(account.get_history()), 1)

    def test_load_dirty_csv_for_checking_account(self):
        account = CheckingAccount("Ivan Ivanov", Decimal("1200"))
        account.account_number = "ACC-100001"

        report = account.load_transactions(CSV_FILE)

        self.assertEqual(report.loaded, 16)
        self.assertEqual(report.skipped, 10)
        self.assertEqual(account.get_balance(), Decimal("4540"))
        self.assertEqual(len(account.get_history()), 16)

    def test_load_dirty_json_for_checking_account(self):
        account = CheckingAccount("Ivan Ivanov", Decimal("1200"))
        account.account_number = "ACC-100001"

        report = account.load_transactions(JSON_FILE)

        self.assertEqual(report.loaded, 16)
        self.assertEqual(report.skipped, 10)
        self.assertEqual(account.get_balance(), Decimal("4540"))
        self.assertEqual(len(account.get_history()), 16)

In [20]:
import unittest


class TestSavingsAccount(unittest.TestCase):

    def test_account_type(self):
        account = SavingsAccount("Ivan Ivanov", Decimal("1000"))

        self.assertEqual(account.account_type, AccountType.SAVINGS)

    def test_withdraw_negative_amount(self):
        account = SavingsAccount("Ivan Ivanov", Decimal("1000"))

        with self.assertRaises(AccountValidationError):
            account.withdraw(Decimal("-100"))

    def test_withdraw_zero_amount(self):
        account = SavingsAccount("Ivan Ivanov", Decimal("1000"))

        with self.assertRaises(AccountValidationError):
            account.withdraw(Decimal("0"))

    def test_withdraw_less_than_50_percent(self):
        account = SavingsAccount("Ivan Ivanov", Decimal("1000"))
        account.withdraw(Decimal("300"))

        self.assertEqual(account.get_balance(), Decimal("700"))

    def test_withdraw_50_percent(self):
        account = SavingsAccount("Ivan Ivanov", Decimal("1000"))
        account.withdraw(Decimal("500"))

        self.assertEqual(account.get_balance(), Decimal("500"))

    def test_withdraw_more_than_50_percent(self):
        account = SavingsAccount("Ivan Ivanov", Decimal("1000"))
        account.withdraw(Decimal("600"))

        self.assertEqual(account.get_balance(), Decimal("1000"))

        operation = account.get_history()[0]
        self.assertEqual(operation["status"], OperationStatus.FAIL)

    def test_apply_interest(self):
        account = SavingsAccount("Ivan Ivanov", Decimal("1000"))
        account.apply_interest(Decimal("10"))

        self.assertEqual(account.get_balance(), Decimal("1100"))

    def test_apply_interest_creates_operation(self):
        account = SavingsAccount("Ivan Ivanov", Decimal("1000"))
        account.apply_interest(Decimal("10"))

        operation = account.get_history()[0]
        self.assertEqual(operation["type"], OperationType.INTEREST)
        self.assertEqual(operation["status"], OperationStatus.SUCCESS)
        self.assertEqual(operation["amount"], Decimal("100"))
        self.assertEqual(operation["current_balance"], Decimal("1100"))

    def test_load_dirty_csv_for_savings_account(self):
        account = SavingsAccount("Ivan Ivanov", Decimal("1500"))
        account.account_number = "ACC-100002"

        report = account.load_transactions(CSV_FILE)

        self.assertEqual(report.loaded, 16)
        self.assertEqual(report.skipped, 11)
        self.assertEqual(account.get_balance(), Decimal("3169.96"))
        self.assertEqual(len(account.get_history()), 16)

    def test_load_dirty_json_for_savings_account(self):
        account = SavingsAccount("Ivan Ivanov", Decimal("1500"))
        account.account_number = "ACC-100002"

        report = account.load_transactions(JSON_FILE)

        self.assertEqual(report.loaded, 16)
        self.assertEqual(report.skipped, 11)
        self.assertEqual(account.get_balance(), Decimal("3169.96"))
        self.assertEqual(len(account.get_history()), 16)



In [21]:
from decimal import Decimal


class TestAccountAnalytics(unittest.TestCase):

    def test_get_largest_operations(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        account.deposit(Decimal("100"))
        account.deposit(Decimal("500"))
        account.withdraw(Decimal("200"))
        account.deposit(Decimal("300"))

        operations = account.get_largest_operations(2)

        self.assertEqual(len(operations), 2)

        self.assertEqual(operations[0]["amount"], Decimal("500"))

        self.assertEqual(operations[1]["amount"], Decimal("300"))

    def test_get_largest_operations_does_not_change_history(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        account.deposit(Decimal("100"))
        account.deposit(Decimal("500"))
        account.withdraw(Decimal("200"))

        original_history = account.get_history()

        account.get_largest_operations(2)

        self.assertEqual(len(account.get_history()), len(original_history))

    def test_get_largest_operations_with_invalid_n(self):
        account = Account("Ivan Ivanov", Decimal("1000"))

        with self.assertRaises(AccountValidationError):
            account.get_largest_operations(0)

        with self.assertRaises(AccountValidationError):
            account.get_largest_operations(-1)

In [22]:
import unittest
from datetime import datetime
from decimal import Decimal


class TestTransactionValidator(unittest.TestCase):

    def setUp(self):
        self.validator = TransactionValidator()

    def test_validate_valid_transaction(self):
        raw_transaction = {
            "account_number": "ACC-100001",
            "account_type": "checking",
            "date": "2025-10-16 22:17:26",
            "operation": "withdraw",
            "amount": "393.0",
            "balance_after": "7431.0",
            "status": "success",
        }

        transaction = self.validator.validate(raw_transaction)

        self.assertEqual(transaction.account_number, "ACC-100001")
        self.assertEqual(transaction.account_type, AccountType.CHECKING)
        self.assertEqual(transaction.operation, OperationType.WITHDRAW)
        self.assertEqual(transaction.status, OperationStatus.SUCCESS)
        self.assertEqual(transaction.amount, Decimal("393.0"))
        self.assertEqual(transaction.balance_after, Decimal("7431.0"))
        self.assertEqual(transaction.date, datetime(2025, 10, 16, 22, 17, 26))

    def test_validate_normalizes_string_values(self):
        raw_transaction = {
            "account_number": " ACC-100001 ",
            "account_type": " CHECKING ",
            "date": "2025-10-16 22:17:26",
            "operation": " WITHDRAW ",
            "amount": "393.0",
            "balance_after": "7431.0",
            "status": " SUCCESS ",
        }

        transaction = self.validator.validate(raw_transaction)

        self.assertEqual(transaction.account_number, "ACC-100001")
        self.assertEqual(transaction.account_type, AccountType.CHECKING)
        self.assertEqual(transaction.operation, OperationType.WITHDRAW)
        self.assertEqual(transaction.status, OperationStatus.SUCCESS)

    def test_validate_negative_amount(self):
        raw_transaction = {
            "account_number": "ACC-100001",
            "account_type": "checking",
            "date": "2025-10-16 22:17:26",
            "operation": "deposit",
            "amount": "-100",
            "balance_after": "900",
            "status": "success",
        }

        with self.assertRaises(TransactionValidationError):
            self.validator.validate(raw_transaction)

    def test_validate_invalid_status(self):
        raw_transaction = {
            "account_number": "ACC-100001",
            "account_type": "checking",
            "date": "2025-10-16 22:17:26",
            "operation": "withdraw",
            "amount": "100",
            "balance_after": "900",
            "status": "succes",
        }

        with self.assertRaises(TransactionValidationError):
            self.validator.validate(raw_transaction)

    def test_validate_invalid_operation(self):
        raw_transaction = {
            "account_number": "ACC-100001",
            "account_type": "checking",
            "date": "2025-10-16 22:17:26",
            "operation": "transfer",
            "amount": "100",
            "balance_after": "900",
            "status": "success",
        }

        with self.assertRaises(TransactionValidationError):
            self.validator.validate(raw_transaction)

    def test_validate_invalid_account_type(self):
        raw_transaction = {
            "account_number": "ACC-100001",
            "account_type": "business",
            "date": "2025-10-16 22:17:26",
            "operation": "deposit",
            "amount": "100",
            "balance_after": "1100",
            "status": "success",
        }

        with self.assertRaises(TransactionValidationError):
            self.validator.validate(raw_transaction)

    def test_validate_invalid_date(self):
        raw_transaction = {
            "account_number": "ACC-100001",
            "account_type": "checking",
            "date": "16/10/2025 22:17",
            "operation": "deposit",
            "amount": "100",
            "balance_after": "1100",
            "status": "success",
        }

        with self.assertRaises(TransactionValidationError):
            self.validator.validate(raw_transaction)

    def test_validate_missing_field(self):
        raw_transaction = {
            "account_number": "ACC-100001",
            "account_type": "checking",
            "date": "2025-10-16 22:17:26",
            "operation": "deposit",
            "amount": "100",
            "balance_after": "1100",
        }

        with self.assertRaises(TransactionValidationError):
            self.validator.validate(raw_transaction)

    def test_validate_invalid_account_number(self):
        raw_transaction = {
            "account_number": "100001",
            "account_type": "checking",
            "date": "2025-10-16 22:17:26",
            "operation": "deposit",
            "amount": "100",
            "balance_after": "1100",
            "status": "success",
        }

        with self.assertRaises(TransactionValidationError):
            self.validator.validate(raw_transaction)

In [23]:
suite = unittest.TestSuite()

suite.addTests(
    unittest.TestLoader().loadTestsFromTestCase(TestAccount)
)

suite.addTests(
    unittest.TestLoader().loadTestsFromTestCase(TestSavingsAccount)
)

suite.addTests(
unittest.TestLoader().loadTestsFromTestCase(TestAccountAnalytics)
)

suite.addTests(
unittest.TestLoader().loadTestsFromTestCase(TestTransactionValidator)
)

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_account_holder (__main__.TestAccount.test_account_holder) ... ok
test_account_number (__main__.TestAccount.test_account_number) ... ok
test_create_account_with_initial_balance (__main__.TestAccount.test_create_account_with_initial_balance) ... ok
test_create_account_with_negative_balance (__main__.TestAccount.test_create_account_with_negative_balance) ... ok
test_create_account_with_zero_balance (__main__.TestAccount.test_create_account_with_zero_balance) ... ok
test_deposit (__main__.TestAccount.test_deposit) ... ok
test_deposit_creates_operation (__main__.TestAccount.test_deposit_creates_operation) ... ok
test_deposit_negative_amount (__main__.TestAccount.test_deposit_negative_amount) ... ok
test_deposit_zero_amount (__main__.TestAccount.test_deposit_zero_amount) ... ok
test_get_history_returns_copy (__main__.TestAccount.test_get_history_returns_copy) ... ok
test_history_contains_operations_in_order (__main__.TestAccount.test_history_contains_operations_in_order) ... ok
test_loa

<unittest.runner.TextTestResult run=40 errors=0 failures=0>